1.pandas的尝试

In [5]:
import pandas as pd
import time

# 记录开始时间
start = time.time()

# 读取 CSV 文件
# ⚠️ 警告：read_csv 会一次性把 3.4GB 全部加载到内存！
# 你的文件没有表头，所以需要指定 header=None 并手动命名列
df = pd.read_csv(
    r"C:\Users\caoruijie\Desktop\UserBehavior.csv",
    header=None,  # 文件没有表头
    names=['user_id', 'item_id', 'merchant_id', 'behavior_type', 'timestamp']  # 手动指定列名
)

# 统计 behavior_type 的分布
result = df['behavior_type'].value_counts()
print(result)

# 打印耗时
print(f"Pandas 耗时：{time.time() - start:.2f} 秒")

behavior_type
pv      89716264
cart     5530446
fav      2888258
buy      2015839
Name: count, dtype: int64
Pandas 耗时：37.81 秒


In [6]:

# ============================================================
# 【第二步】Polars - Lazy API 流式执行 (核外计算)
# ============================================================
import polars as pl
import time

start = time.time()

q = pl.scan_csv(
    r"C:\Users\caoruijie\Desktop\UserBehavior.csv",
    has_header=False
)

q = q.with_columns([
    pl.col("column_4").alias("behavior_type")
])

result = q.group_by("behavior_type").agg(pl.count())

df_result = result.collect(streaming=True)

print(df_result)
print(f"Polars 耗时：{time.time() - start:.2f} 秒")


# ============================================================
# 【第三步】DuckDB - SQL 直接查询磁盘文件 (核外计算)
# ============================================================
import duckdb

start = time.time()

# 修正：DuckDB 列名是 column0, column1, column2, column3, column4
query = """
    SELECT column4 as behavior_type, COUNT(*) as count 
    FROM read_csv_auto('C:/Users/caoruijie/Desktop/UserBehavior.csv', header=false)
    GROUP BY behavior_type
"""

result = duckdb.query(query).df()
print(result)
print(f"DuckDB 耗时：{time.time() - start:.2f} 秒")

C:\Users\caoruijie\AppData\Local\Temp\ipykernel_25480\591756419.py:18: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  result = q.group_by("behavior_type").agg(pl.count())
C:\Users\caoruijie\AppData\Local\Temp\ipykernel_25480\591756419.py:20: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_result = result.collect(streaming=True)


shape: (4, 2)
┌───────────────┬──────────┐
│ behavior_type ┆ count    │
│ ---           ┆ ---      │
│ str           ┆ u32      │
╞═══════════════╪══════════╡
│ pv            ┆ 89716264 │
│ fav           ┆ 2888258  │
│ buy           ┆ 2015839  │
│ cart          ┆ 5530446  │
└───────────────┴──────────┘
Polars 耗时：3.76 秒
        behavior_type  count
0          1511933111    192
1          1511589033    144
2          1511623664    178
3          1511666744    138
4          1512089425    103
...               ...    ...
815854     1511531953      1
815855     1511521009      1
815856     1511422161      1
815857     1511536977      1
815858     1511435292      1

[815859 rows x 2 columns]
DuckDB 耗时：4.79 秒


任务三：

In [7]:


# ============================================================
# 任务 3：业务数据的深度"体检"
# ============================================================

import duckdb
import time

FILE_PATH = r"C:\Users\caoruijie\Desktop\UserBehavior.csv"

# ==================== 1. 缺失值探查 ====================
print("【1】缺失值探查")
start = time.time()

query = """
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) as user_id_null,
        SUM(CASE WHEN behavior_type IS NULL OR behavior_type = '' THEN 1 ELSE 0 END) as behavior_type_null
    FROM read_csv('{file}', 
                  columns={{'user_id': 'INTEGER', 'item_id': 'INTEGER', 'merchant_id': 'INTEGER', 
                           'behavior_type': 'VARCHAR', 'timestamp': 'INTEGER'}})
""".format(file=FILE_PATH)

print(duckdb.query(query).df())
print(f"耗时：{time.time() - start:.2f} 秒\n")


# ==================== 2. 时间异常诊断 ====================
print("【2】时间异常诊断")
start = time.time()

query = """
    SELECT 
        MIN(timestamp) as min_ts,
        MAX(timestamp) as max_ts,
        COUNT(CASE WHEN timestamp <= 0 THEN 1 END) as invalid_1970_count
    FROM read_csv('{file}',
                  columns={{'user_id': 'INTEGER', 'item_id': 'INTEGER', 'merchant_id': 'INTEGER',
                           'behavior_type': 'VARCHAR', 'timestamp': 'INTEGER'}})
""".format(file=FILE_PATH)

result = duckdb.query(query).df()
print(result)

# 用 DuckDB 直接转换日期
query_date = """
    SELECT 
        MIN(to_timestamp(timestamp)) as min_date,
        MAX(to_timestamp(timestamp)) as max_date
    FROM read_csv('{file}',
                  columns={{'user_id': 'INTEGER', 'item_id': 'INTEGER', 'merchant_id': 'INTEGER',
                           'behavior_type': 'VARCHAR', 'timestamp': 'INTEGER'}})
    WHERE timestamp > 0
""".format(file=FILE_PATH)

date_result = duckdb.query(query_date).df()
print(f"最小时间：{date_result['min_date'][0]}")
print(f"最大时间：{date_result['max_date'][0]}")
print(f"1970 年无效数据：{result['invalid_1970_count'][0]:,} 条")
print(f"耗时：{time.time() - start:.2f} 秒\n")


# ==================== 3. 独立访客 UV ====================
print("【3】独立访客 UV")
start = time.time()

query = """
    SELECT COUNT(DISTINCT user_id) as uv_count
    FROM read_csv('{file}',
                  columns={{'user_id': 'INTEGER', 'item_id': 'INTEGER', 'merchant_id': 'INTEGER',
                           'behavior_type': 'VARCHAR', 'timestamp': 'INTEGER'}})
""".format(file=FILE_PATH)

result = duckdb.query(query).df()
print(f"UV = {result['uv_count'][0]:,} 个")
print(f"耗时：{time.time() - start:.2f} 秒\n")


# ==================== 4. 昼夜作息规律（24 小时分布 + 购买高峰） ====================
print("【4】昼夜作息规律")
start = time.time()

query = """
    SELECT 
        EXTRACT(HOUR FROM to_timestamp(timestamp)) as hour,
        behavior_type,
        COUNT(*) as cnt
    FROM read_csv('{file}',
                  columns={{'user_id': 'INTEGER', 'item_id': 'INTEGER', 'merchant_id': 'INTEGER',
                           'behavior_type': 'VARCHAR', 'timestamp': 'INTEGER'}})
    WHERE behavior_type IN ('pv', 'buy') AND timestamp > 0
    GROUP BY hour, behavior_type
    ORDER BY hour, behavior_type
""".format(file=FILE_PATH)

result = duckdb.query(query).df()
print(result)

# 购买高峰期
buy_data = result[result['behavior_type'] == 'buy']
peak = buy_data.loc[buy_data['cnt'].idxmax()]
print(f"\n购买高峰期：{int(peak['hour'])}:00-{int(peak['hour'])+1}:00, 次数：{peak['cnt']:,}")
print(f"耗时：{time.time() - start:.2f} 秒\n")

【1】缺失值探查
   total_rows  user_id_null  behavior_type_null
0   100150807           0.0                 0.0
耗时：2.46 秒

【2】时间异常诊断
       min_ts      max_ts  invalid_1970_count
0 -2134949234  2122867355                 318
最小时间：1970-01-01 08:04:19+08:00
最大时间：2037-04-09 13:22:35+08:00
1970 年无效数据：318 条
耗时：4.27 秒

【3】独立访客 UV
UV = 987,994 个
耗时：2.03 秒

【4】昼夜作息规律
    hour behavior_type      cnt
0      0           buy    57776
1      0            pv  3058113
2      1           buy    23169
3      1            pv  1422174
4      2           buy    12012
5      2            pv   769511
6      3           buy     8026
7      3            pv   525196
8      4           buy     6748
9      4            pv   449865
10     5           buy     8135
11     5            pv   581844
12     6           buy    18014
13     6            pv  1226971
14     7           buy    37679
15     7            pv  2229521
16     8           buy    64917
17     8            pv  3043120
18     9           buy    96134
19   

任务四


In [ ]:
# ============================================================
# 任务 4：数据列式固化与存储优化（Parquet 格式转换 + 分区落盘）
# ============================================================

import duckdb
import time
import os

FILE_PATH = r"C:\Users\caoruijie\Desktop\UserBehavior.csv"
OUTPUT_DIR = r"G:\Users\caoruijie\big data\clean_data_partitioned"

# ==================== 1. 定义业务清洗规则 ====================
print("=" * 60)
print("【1】定义清洗规则并转换 Parquet（按 behavior_type 分区）")
print("=" * 60)
print(f"输出目录：{OUTPUT_DIR}")

start = time.time()

# 创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 使用 DuckDB 直接写入分区 Parquet 文件
# 清洗规则：
#   - 剔除 timestamp <= 0 的 1970 年无效数据
#   - 剔除 behavior_type 为空的数据
query = """
    COPY (
        SELECT 
            user_id,
            item_id,
            merchant_id,
            behavior_type,
            timestamp
        FROM read_csv('{file}',
                      columns={{'user_id': 'INTEGER', 'item_id': 'INTEGER', 'merchant_id': 'INTEGER',
                               'behavior_type': 'VARCHAR', 'timestamp': 'INTEGER'}})
        WHERE timestamp > 0 
          AND behavior_type IS NOT NULL 
          AND behavior_type != ''
    )
    TO '{output}'
    (FORMAT 'parquet', PARTITION_BY 'behavior_type')
""".format(file=FILE_PATH, output=OUTPUT_DIR)

duckdb.execute(query)

elapsed = time.time() - start
print(f"✅ Parquet 分区文件生成完成!")
print(f"耗时：{elapsed:.2f} 秒\n")


# ==================== 2. 查看分区结果 ====================
print("=" * 60)
print("【2】分区目录结构")
print("=" * 60)

# 遍历输出目录，展示分区结构
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = ' ' * 2 * (level + 1)
    for file in files[:5]:  # 每个目录只显示前 5 个文件
        print(f'{sub_indent}{file}')
    if len(files) > 5:
        print(f'{sub_indent}... 共 {len(files)} 个文件')


# ==================== 3. 统计各分区大小 ====================
print("\n" + "=" * 60)
print("【3】各分区文件大小统计")
print("=" * 60)

import glob

parquet_files = glob.glob(f"{OUTPUT_DIR}/**/*.parquet", recursive=True)
total_size = 0

print(f"{'分区':<20} {'文件数':<10} {'大小 (MB)':<15}")
print("-" * 45)

for behavior in ['pv', 'buy', 'cart', 'fav']:
    partition_path = f"{OUTPUT_DIR}/behavior_type={behavior}"
    files = glob.glob(f"{partition_path}/*.parquet")
    if files:
        size_mb = sum(os.path.getsize(f) for f in files) / (1024 * 1024)
        total_size += size_mb
        print(f"{behavior:<20} {len(files):<10} {size_mb:<15.2f}")

print("-" * 45)
print(f"{'总计':<20} {'':<10} {total_size:<15.2f} MB ({total_size/1024:.2f} GB)")

# 与原始文件对比
original_size = os.path.getsize(FILE_PATH) / (1024 * 1024)
compression_ratio = (1 - total_size / original_size) * 100
print(f"\n原始 CSV 大小：{original_size:.2f} MB")
print(f"Parquet 大小：{total_size:.2f} MB")
print(f"压缩率：{compression_ratio:.1f}% (节省空间)")


# ==================== 4. 验证分区数据 ====================
print("\n" + "=" * 60)
print("【4】验证各分区数据量")
print("=" * 60)

query_verify = """
    SELECT 
        behavior_type,
        COUNT(*) as cnt
    FROM read_parquet('{output}/**/*.parquet')
    GROUP BY behavior_type
    ORDER BY cnt DESC
""".format(output=OUTPUT_DIR)

result = duckdb.query(query_verify).df()
print(result)

total_rows = result['cnt'].sum()
print(f"\n清洗后总行数：{total_rows:,}")
print(f"原始总行数：100,150,807")
print(f"剔除行数：{100150807 - total_rows:,} (主要是 318 条 1970 年无效数据)")

print(f"\n耗时：{time.time() - start:.2f} 秒")

【1】定义清洗规则并转换 Parquet（按 behavior_type 分区）
输出目录：G:\Users\caoruijie\big data\clean_data_partitioned


IOException: IO Error: Directory "G:\Users\caoruijie\big data\clean_data_partitioned" is not empty! Enable OVERWRITE option to overwrite files